In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 1. Define Dataset: Input sentences and labels
# Now the labels are not just "<eos>" but other words as well.
training_data = [
    ("i", "love"),
    ("i love", "pytorch"),
    ("i love pytorch", "<eos>"),
    ("the", "cat"),
    ("the cat", "is"),
    ("the cat is", "sleeping"),
    ("the cat is sleeping", "<eos>"),
    ("we are", "learning"),
    ("we are learning", "ml"),
    ("we are learning ml", "<eos>")
]

# 2. Create Vocabulary and Index Mapping
all_words = set()
for sentence, label in training_data:
    all_words.update(sentence.split())
    all_words.add(label)
all_words.add("<pad>")
vocab = {word: i for i, word in enumerate(sorted(list(all_words)))}
vocab_size = len(vocab)
seq_len = max(len(data[0].split()) for data in training_data) + 1

# 3. Define Dataset Class
class TextDataset(Dataset):
    def __init__(self, data, vocab, seq_len):
        self.data = data
        self.vocab = vocab
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence, label = self.data[idx]
        sentence_words = sentence.split()

        # Convert sentence to integer indices
        input_indices = [self.vocab[word] for word in sentence_words]

        # Add padding tokens if sentence is shorter than max length
        if len(input_indices) < self.seq_len:
            padding_needed = self.seq_len - len(input_indices)
            input_indices.extend([self.vocab["<pad>"]] * padding_needed)
        # Truncate if sentence is longer
        else:
            input_indices = input_indices[:self.seq_len]

        # Convert label to integer index
        label_index = self.vocab[label]

        return torch.tensor(input_indices), torch.tensor(label_index)

# 4. Define Model (Improved)
class SimpleTransformerModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, seq_len, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(seq_len, embedding_dim)
        self.multihead_attention = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads, batch_first=True, dropout=dropout)
        
        # Feed-Forward Network
        self.linear1 = nn.Linear(embedding_dim, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, embedding_dim)

        # Layer Normalization
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.linear_out = nn.Linear(embedding_dim, vocab_size)
        self.pad_idx = vocab.get("<pad>", 0) # Get padding index

    def forward(self, x):
        # Create padding mask
        # Shape: (batch_size, seq_len)
        key_padding_mask = (x == self.pad_idx)

        batch_size, current_seq_len = x.shape
        word_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(torch.arange(current_seq_len, device=x.device))
        
        # Add positional embedding and apply dropout
        input_emb = self.dropout1(word_emb + pos_emb)

        # Self-Attention with Layer Normalization
        attn_output, _ = self.multihead_attention(input_emb, input_emb, input_emb, key_padding_mask=key_padding_mask)
        x = self.norm1(input_emb + self.dropout2(attn_output))

        # Feed-Forward Network with Layer Normalization
        ff_output = self.linear2(self.dropout(torch.relu(self.linear1(x))))
        x = self.norm2(x + ff_output)

        last_token_output = x[:, -1, :]
        output = self.linear_out(last_token_output)
        return output

# 5. Define Model, Loss Function, and Optimizer
embedding_dim = 16
num_heads = 2
model = SimpleTransformerModel(vocab_size, embedding_dim, num_heads, seq_len)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 6. Train with DataLoader
dataset = TextDataset(training_data, vocab, seq_len)
dataloader = DataLoader(dataset, batch_size=len(training_data), shuffle=True)

num_epochs = 200
print("Starting training...")
for epoch in range(num_epochs):
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        prediction = model(inputs)
        loss = loss_fn(prediction, labels)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

print("\nTraining complete!")

# --- [ Text Generation using the Trained Model ] ---

# 7. Define Text Generation Function
def generate_sentence(model, initial_prompt, vocab, word_by_index, seq_len, max_len=10):
    model.eval() # Set model to evaluation mode
    generated_sentence = initial_prompt.split()

    print(f"\nInitial prompt: '{initial_prompt}'")
    print("Generation process:")

    # Loop until max length or <eos> token is predicted
    for i in range(max_len):
        input_words = generated_sentence

        # Prepare input sentence for the model
        padded_input_words = input_words[:seq_len]
        while len(padded_input_words) < seq_len:
            padded_input_words.append("<pad>")

        input_indices = [vocab[word] for word in padded_input_words]
        input_tensor = torch.tensor([input_indices])

        # Get model's prediction
        with torch.no_grad():
            prediction = model(input_tensor)

        predicted_index = torch.argmax(prediction, dim=1).item()
        predicted_word = word_by_index[predicted_index]

        # Stop if <eos> token is predicted
        if predicted_word == "<eos>":
            break

        # Append the new word and print
        generated_sentence.append(predicted_word)
        print(f"-> Predicted next word: '{predicted_word}'")

    final_sentence = ' '.join(generated_sentence)
    print(f"\nFinal generated sentence: '{final_sentence}'")
    return final_sentence

# 8. Loop for continuous user input
print("\n--- Text Generation ---")
word_by_index = {i: word for word, i in vocab.items()}

# Print list of trained sentences
print("\n--- Trained Sentences ---")
for sentence, label in training_data:
    print(f"- '{sentence}' -> '{label}'")

while True:
    try:
        user_input = input("\nEnter a starting sentence ('exit' to quit): ")

        if user_input.lower() == 'exit':
            print("Exiting program.")
            break

        if not user_input.strip():
            print("Please enter a valid sentence.")
            continue

        generate_sentence(model, user_input, vocab, word_by_index, seq_len)

    except KeyError as e:
        print(f"Error: Word '{e}' not in vocabulary. Please enter a sentence with words from the training set.")
    except Exception as e:
        print(f"An error occurred: {e}")